# Initial Analysis
To see what content is present in swe_bench

## Data Fields
1. **problem_statement**: The title and body of the GitHub issue. For this work, this is the primary input signal—the natural language description of the bug.

2. **base_commit**: The commit hash of the repository before the bug fix was applied. This represents the buggy version of the code that you will analyze.

3. **repo**: The name of the source repository (e.g., matplotlib/matplotlib).

4. **patch**: The golden code patch that resolved the issue. This represents the ground truth fix for the bug. While your goal is to localize the bug, this field tells you exactly which lines were changed to fix it.

5. **FAIL_TO_PASS**:  A JSON list of tests that were failing before the patch and passed after it was applied. This is a critical field for you. It provides a direct, executable link between the bug report (problem_statement) and the specific parts of the codebase that were malfunctioning. You can use these failing tests to validate the location of a bug.

6. **PASS_TO_PASS**: A list of tests that passed both before and after the fix, used for regression testing.

7. **test_patch**: A patch file for new tests that were added as part of the pull request.

8. **hints_text**: Comments on the issue made before the pull request was created. This provides additional natural language context about the bug.

9. **instance_id**

10. **created_at** 

11. **version** 

12. **environment_setup_commit**

In [25]:
import pandas as pd
import re
import os

In [2]:
# Provide the path to your data file.
# We will first start with 'dev' split since it's the smallest.
dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/dev-00000-of-00001.parquet'

In [4]:
df = pd.read_parquet(dev_file_path)

In [5]:
print(" ----- Basic DataFrame Info -------")
# It display column names, non-null counts and data types
df.info

 ----- Basic DataFrame Info -------


<bound method DataFrame.info of                   repo              instance_id  \
0    sqlfluff/sqlfluff  sqlfluff__sqlfluff-4764   
1    sqlfluff/sqlfluff  sqlfluff__sqlfluff-2862   
2    sqlfluff/sqlfluff  sqlfluff__sqlfluff-2336   
3    sqlfluff/sqlfluff  sqlfluff__sqlfluff-5074   
4    sqlfluff/sqlfluff  sqlfluff__sqlfluff-3436   
..                 ...                      ...   
220    pydicom/pydicom     pydicom__pydicom-809   
221    pydicom/pydicom     pydicom__pydicom-933   
222    pydicom/pydicom    pydicom__pydicom-1633   
223    pydicom/pydicom    pydicom__pydicom-1428   
224    pydicom/pydicom    pydicom__pydicom-1256   

                                  base_commit  \
0    a820c139ccbe6d1865d73c4a459945cd69899f8f   
1    447ecf862a4d2b977d0add9f444655357b9c4f1f   
2    37a993f7ad841ab3035d1db5ce6525f2e5584fd5   
3    7b7fd603a19755a9f3707ebbf95d18ee635716d8   
4    23cd31e77a712a210c734e38488d7a34afd83a25   
..                                        ...   
220  356a51a

In [7]:
print("First 3 rows of the DataFrame")
print(df.head(1))

First 3 rows of the DataFrame
                repo              instance_id  \
0  sqlfluff/sqlfluff  sqlfluff__sqlfluff-4764   

                                base_commit  \
0  a820c139ccbe6d1865d73c4a459945cd69899f8f   

                                               patch  \
0  diff --git a/src/sqlfluff/cli/commands.py b/sr...   

                                          test_patch  \
0  diff --git a/test/cli/commands_test.py b/test/...   

                                   problem_statement hints_text  \
0  Enable quiet mode/no-verbose in CLI for use in...              

             created_at version  \
0  2023-04-16T14:24:42Z     1.4   

                                        FAIL_TO_PASS  \
0  ["test/cli/commands_test.py::test__cli__fix_mu...   

                                        PASS_TO_PASS  \
0  ["test/cli/commands_test.py::test__cli__comman...   

                   environment_setup_commit  
0  d19de0ecd16d298f9e3bfb91da122734c40c01e5  


In [8]:
# Getting all column names
print(df.columns)

Index(['repo', 'instance_id', 'base_commit', 'patch', 'test_patch',
       'problem_statement', 'hints_text', 'created_at', 'version',
       'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit'],
      dtype='object')


In [9]:
# Inspecting a single record patch
first_patch = df.iloc[0]['patch']
print(f"instance_id: {df.iloc[0]['instance_id']}")
print(f"first_patch: {first_patch}")

instance_id: sqlfluff__sqlfluff-4764
first_patch: diff --git a/src/sqlfluff/cli/commands.py b/src/sqlfluff/cli/commands.py
--- a/src/sqlfluff/cli/commands.py
+++ b/src/sqlfluff/cli/commands.py
@@ -44,6 +44,7 @@
     dialect_selector,
     dialect_readout,
 )
+from sqlfluff.core.linter import LintingResult
 from sqlfluff.core.config import progress_bar_configuration
 
 from sqlfluff.core.enums import FormatType, Color
@@ -691,12 +692,16 @@ def lint(
         sys.exit(EXIT_SUCCESS)
 
 
-def do_fixes(lnt, result, formatter=None, **kwargs):
+def do_fixes(
+    result: LintingResult, formatter: Optional[OutputStreamFormatter] = None, **kwargs
+):
     """Actually do the fixes."""
-    click.echo("Persisting Changes...")
+    if formatter and formatter.verbosity >= 0:
+        click.echo("Persisting Changes...")
     res = result.persist_changes(formatter=formatter, **kwargs)
     if all(res.values()):
-        click.echo("Done. Please check your files to confirm.")
+        if formatter and

## Helper Functions
In this section we will add helper methods to extract relevant information from the dataset

In [3]:
import re

In [26]:
def extract_file_paths_from_patch(patch_text):
    '''
    Parses a patch string which is column in the dataset to extract unique file paths.

    Args:
        patch_text (str): The full text content of a patch.
    
    Returns:
        list: A list of unique file paths found in the patch.
    '''

    # We are creating a regex pattern which looks for lines starting with '--- a/' or '+++ b/'
    # and captuires the file path that follows. It also handles file paths that may contain spaces.
    regex = r"^(?:--- a\/|\+\+\+ b\/)(.+?)\t*$"

    # Find all matches in the text
    matches = re.findall(regex, patch_text, re.MULTILINE)

    # Return a list of unique file paths
    return list(set(matches))

In [12]:
# Test Example:
sample_patch = """
diff --git a/src/sqlfluff/cli/commands.py b/src/sqlfluff/cli/commands.py
--- a/src/sqlfluff/cli/commands.py
+++ b/src/sqlfluff/cli/commands.py
@@ -44,6 +44,7 @@
# ... content of patch ...
diff --git a/src/sqlfluff/cli/formatters.py b/src/sqlfluff/cli/formatters.py
--- a/src/sqlfluff/cli/formatters.py
+++ b/src/sqlfluff/cli/formatters.py
@@ -94,7 +94,7 @@ def __init__(
# ... content of patch ...
diff --git a/src/sqlfluff/core/linter/linted_dir.py b/src/sqlfluff/core/linter/linted_dir.py
--- a/src/sqlfluff/core/linter/linted_dir.py
+++ b/src/sqlfluff/core/linter/linted_dir.py
@@ -117,7 +117,11 @@ def persist_changes(
"""

In [13]:
# Extract the file paths from the sample patch
ground_truth_files = extract_file_paths_from_patch(sample_patch)
print("Extracted Ground Truth Files: ")
print(ground_truth_files)

Extracted Ground Truth Files: 
['src/sqlfluff/core/linter/linted_dir.py', 'src/sqlfluff/cli/formatters.py', 'src/sqlfluff/cli/commands.py']


# Applying on whole DataFrame
The idea of applying all transformation through helper method on full dataset, i.e. dataframe

In [14]:
df['ground_truth_files'] = df['patch'].apply(extract_file_paths_from_patch)

In [15]:
# Now view the all the fields which we require for our dataset.
print(df[['repo', 'instance_id', 'problem_statement', 'base_commit', 'ground_truth_files']].head())

                repo              instance_id  \
0  sqlfluff/sqlfluff  sqlfluff__sqlfluff-4764   
1  sqlfluff/sqlfluff  sqlfluff__sqlfluff-2862   
2  sqlfluff/sqlfluff  sqlfluff__sqlfluff-2336   
3  sqlfluff/sqlfluff  sqlfluff__sqlfluff-5074   
4  sqlfluff/sqlfluff  sqlfluff__sqlfluff-3436   

                                   problem_statement  \
0  Enable quiet mode/no-verbose in CLI for use in...   
1  fix keep adding new line on wrong place \n### ...   
2  L026: Rule incorrectly flag column does not ex...   
3  Inconsistent output depending on --processes f...   
4  Fatal templating error with Jinja templater. T...   

                                base_commit  \
0  a820c139ccbe6d1865d73c4a459945cd69899f8f   
1  447ecf862a4d2b977d0add9f444655357b9c4f1f   
2  37a993f7ad841ab3035d1db5ce6525f2e5584fd5   
3  7b7fd603a19755a9f3707ebbf95d18ee635716d8   
4  23cd31e77a712a210c734e38488d7a34afd83a25   

                                  ground_truth_files  
0  [src/sqlfluff/core/linter/l

In [17]:
print(f"Repository name: {df.iloc[100]['repo']}")
print(f"Bug ID: {df.iloc[100]['instance_id']}")
print(f"Bug Description: {df.iloc[100]['problem_statement']}")
print(f"Base Commit ID: {df.iloc[100]['base_commit']}")
print(f"Ground Truth Files: {df.iloc[100]['ground_truth_files']}")

Repository name: pvlib/pvlib-python
Bug ID: pvlib__pvlib-python-1181
Bug Description: remove ModelChain.orientation_strategy
I don't like that `ModelChain(system, location, orientation_strategy='flat`|`south_at_latitude_tilt`) modifies the `system` object. It's not something we do anywhere else in pvlib. `orientation_strategy` only supports flat and south_at_latitude_tilt, neither of which are commonly used in the real world in 2020. 

I think we should remove it, maybe even without deprecation, in 0.8.

I'm ok with keeping the `modelchain.get_orientation` function for now.

Base Commit ID: 8b98768818ee5ad85d9479877533651a2e9dc2cd
Ground Truth Files: ['pvlib/modelchain.py']


## Extracting unique repository names from the complete dataset
The aim is to extract repo names

In [23]:
# loads all train, dev, and test parquet files from a directory, combines them, and finds 
# unique repositories
dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/dev-00000-of-00001.parquet'
train_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/train-00000-of-00001.parquet'
test_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/test-00000-of-00001.parquet'


In [24]:
# load the data in pandas dataframe
df_dev = pd.read_parquet(dev_file_path)
print("Dev samples shape: ", df_dev.shape)
df_train = pd.read_parquet(train_file_path)
print("Train samples shape: ", df_train.shape)
df_test = pd.read_parquet(test_file_path)
print("Test samples shape: ", df_test.shape)

df = pd.concat([df_train, df_dev, df_test])
print("Combined shape: ", df.shape)

Dev samples shape:  (225, 12)
Train samples shape:  (19008, 12)
Test samples shape:  (2294, 12)
Combined shape:  (21527, 12)


In [22]:
# Get unique values from the 'repo' column
unique_repositories = df_test['repo'].unique()
print(f"Number of unique repositories: {len(unique_repositories)}")
print("List of unique repositories")
for repo in sorted(unique_repositories):
    print(repo)

Number of unique repositories: 12
List of unique repositories
astropy/astropy
django/django
matplotlib/matplotlib
mwaskom/seaborn
pallets/flask
psf/requests
pydata/xarray
pylint-dev/pylint
pytest-dev/pytest
scikit-learn/scikit-learn
sphinx-doc/sphinx
sympy/sympy


## Repository - Number of Bug Reports per Repo - Duplicate Bug Reports

In [6]:
# loads all train, dev, and test parquet files from a directory, combines them, and finds 
# unique repositories
swe_bench_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/dev-00000-of-00001.parquet'
swe_bench_train_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/train-00000-of-00001.parquet'
swe_bench_test_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/test-00000-of-00001.parquet'


In [7]:
# load the data in pandas dataframe
swe_bench_dev = pd.read_parquet(swe_bench_dev_file_path)
print("Dev samples shape: ", swe_bench_dev.shape)
swe_bench_train = pd.read_parquet(swe_bench_train_file_path)
print("Train samples shape: ", swe_bench_train.shape)
swe_bench_test = pd.read_parquet(swe_bench_test_file_path)
print("Test samples shape: ", swe_bench_test.shape)


Dev samples shape:  (225, 12)
Train samples shape:  (19008, 12)
Test samples shape:  (2294, 12)


### For Dev Split

In [9]:
# Step 1: Count total bug reports per repo per different splits (train, test, dev)
# Group by 'repo' and count the number of instances
bug_counts = swe_bench_dev.groupby('repo')['instance_id'].count().reset_index()
bug_counts.rename(columns={'instance_id': 'Total_Bug_Reports'}, inplace=True)

# Step 2: Count duplicate bug reports per repo
# A duplicate is a non-unique 'problem statement' within a repo group.
# The number of duplocates is the total rows minus the number of unique rows.
duplicate_counts = swe_bench_dev.groupby('repo').apply(
    lambda x: x.shape[0] - x['problem_statement'].nunique()
).reset_index(name='Duplicate_Bug_Reports')

# Step 3: Combine and sort the results
# Merge two dataframes on the 'repo' column
results_df = pd.merge(bug_counts, duplicate_counts, on='repo')

# Sort the final Dataframe in descending order by the number of bug reports
sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False)

# Reset index for cleaner presentation
sorted_results.reset_index(drop=True, inplace=True)

print("Dev Split Analysis")
print(sorted_results.to_string())

Dev Split Analysis
                           repo  Total_Bug_Reports  Duplicate_Bug_Reports
0            pvlib/pvlib-python                 63                      1
1               pydicom/pydicom                 56                      0
2             sqlfluff/sqlfluff                 50                      0
3            pylint-dev/astroid                 31                      2
4               pyvista/pyvista                 16                      0
5  marshmallow-code/marshmallow                  9                      0


/tmp/ipykernel_30586/1244931687.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  duplicate_counts = swe_bench_dev.groupby('repo').apply(


### For Test Split

In [12]:
# Step 1: Count total bug reports per repo per different splits (train, test, dev)
# Group by 'repo' and count the number of instances
bug_counts = swe_bench_test.groupby('repo')['instance_id'].count().reset_index()
bug_counts.rename(columns={'instance_id': 'Total_Bug_Reports'}, inplace=True)

# Step 2: Count duplicate bug reports per repo
# A duplicate is a non-unique 'problem statement' within a repo group.
# The number of duplocates is the total rows minus the number of unique rows.
duplicate_counts = swe_bench_test.groupby('repo').apply(
    lambda x: x.shape[0] - x['problem_statement'].nunique()
).reset_index(name='Duplicate_Bug_Reports')

# Step 3: Combine and sort the results
# Merge two dataframes on the 'repo' column
results_df = pd.merge(bug_counts, duplicate_counts, on='repo')

# Sort the final Dataframe in descending order by the number of bug reports
sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False)

# Reset index for cleaner presentation
sorted_results.reset_index(drop=True, inplace=True)

print("Test Split Analysis")
print(sorted_results.to_string())

Test Split Analysis
                         repo  Total_Bug_Reports  Duplicate_Bug_Reports
0               django/django                850                      6
1                 sympy/sympy                386                      2
2   scikit-learn/scikit-learn                229                      1
3           sphinx-doc/sphinx                187                      3
4       matplotlib/matplotlib                184                      1
5           pytest-dev/pytest                119                      3
6               pydata/xarray                110                      0
7             astropy/astropy                 95                      2
8           pylint-dev/pylint                 57                      0
9                psf/requests                 44                      1
10            mwaskom/seaborn                 22                      0
11              pallets/flask                 11                      0


/tmp/ipykernel_30586/3572654724.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  duplicate_counts = swe_bench_test.groupby('repo').apply(


### For Train Split

In [13]:
# Step 1: Count total bug reports per repo per different splits (train, test, dev)
# Group by 'repo' and count the number of instances

print(f"Number of unique repositories: {len(swe_bench_train['repo'].unique())}")
bug_counts = swe_bench_train.groupby('repo')['instance_id'].count().reset_index()
bug_counts.rename(columns={'instance_id': 'Total_Bug_Reports'}, inplace=True)

# Step 2: Count duplicate bug reports per repo
# A duplicate is a non-unique 'problem statement' within a repo group.
# The number of duplocates is the total rows minus the number of unique rows.
duplicate_counts = swe_bench_train.groupby('repo').apply(
    lambda x: x.shape[0] - x['problem_statement'].nunique()
).reset_index(name='Duplicate_Bug_Reports')

# Step 3: Combine and sort the results
# Merge two dataframes on the 'repo' column
results_df = pd.merge(bug_counts, duplicate_counts, on='repo')

# Sort the final Dataframe in descending order by the number of bug reports
sorted_results = results_df.sort_values(by='Total_Bug_Reports', ascending=False)

# Reset index for cleaner presentation
sorted_results.reset_index(drop=True, inplace=True)

print("Train Split Analysis")
print(sorted_results.to_string())

Number of unique repositories: 35
Train Split Analysis
                              repo  Total_Bug_Reports  Duplicate_Bug_Reports
0                pandas-dev/pandas               5049                     85
1                    Qiskit/qiskit               1406                     70
2         huggingface/transformers               1058                     36
3                 mesonbuild/meson                954                     17
4                      numpy/numpy                937                    148
5   googleapis/google-cloud-python                926                     30
6                 pantsbuild/pants                900                    201
7                   conan-io/conan                855                     12
8                  ipython/ipython                850                     18
9                         pypa/pip                686                     10
10                     conda/conda                629                     22
11                  d

/tmp/ipykernel_30586/1144393295.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  duplicate_counts = swe_bench_train.groupby('repo').apply(


### Unique repos in combined dataset 
To confirm that uniques are not present in other splits

In [14]:
swe_bench_combined = pd.concat([swe_bench_train, swe_bench_test, swe_bench_dev])
print(f"Number of unique repositories: {len(swe_bench_combined['repo'].unique())}")

Number of unique repositories: 53


## Combining all splits and extracting total bug reports, duplicate bug reports, and total unique reports

In [27]:
swe_bench_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/dev-00000-of-00001.parquet'
swe_bench_train_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/train-00000-of-00001.parquet'
swe_bench_test_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/test-00000-of-00001.parquet'

In [28]:
# load the data in pandas dataframe
swe_bench_dev = pd.read_parquet(swe_bench_dev_file_path)
print("Dev samples shape: ", swe_bench_dev.shape)
swe_bench_train = pd.read_parquet(swe_bench_train_file_path)
print("Train samples shape: ", swe_bench_train.shape)
swe_bench_test = pd.read_parquet(swe_bench_test_file_path)
print("Test samples shape: ", swe_bench_test.shape)

Dev samples shape:  (225, 12)
Train samples shape:  (19008, 12)
Test samples shape:  (2294, 12)


In [29]:
# Combine the train, dev, and test DataFrames
swe_bench_all = pd.concat([swe_bench_train, swe_bench_dev, swe_bench_test], ignore_index=True)

In [18]:
# Pre-processing - create the repository link from the repo column
swe_bench_combined['repo_link'] = swe_bench_combined['repo'].apply(
    lambda x: f"https://github.com/{x}"
)

In [20]:
# Group by repo and perform all aggregations
# We count total instances and the number of unique problem statements
repo_analysis = swe_bench_combined.groupby('repo').agg(
    Total_Bug_Reports=('instance_id', 'count'),
    Unique_Problem_Statements=('problem_statement', 'nunique'),
    repo_link=('repo_link', 'first')
).reset_index()

# Calculate duplicate and unique bug report columns
repo_analysis['Duplicate_Bug_Reports'] = repo_analysis['Total_Bug_Reports'] - repo_analysis['Unique_Problem_Statements']
repo_analysis['Unique_Bug_Reports'] = repo_analysis['Unique_Problem_Statements']

# Finalize the dataframe for presentation, select and reorder the column for the final output
final_columns = [
    'repo', 'repo_link', 'Total_Bug_Reports', 'Duplicate_Bug_Reports', 'Unique_Bug_Reports'
]
sorted_results = repo_analysis[final_columns].sort_values(
    by='Total_Bug_Reports', ascending=False
).reset_index(drop=True)

# Print the final combined analysis
print("---Combined Swe-Bench Analysis")
print(sorted_results.to_string())

---Combined Swe-Bench Analysis
                              repo                                          repo_link  Total_Bug_Reports  Duplicate_Bug_Reports  Unique_Bug_Reports
0                pandas-dev/pandas               https://github.com/pandas-dev/pandas               5049                     85                4964
1                    Qiskit/qiskit                   https://github.com/Qiskit/qiskit               1406                     70                1336
2         huggingface/transformers        https://github.com/huggingface/transformers               1058                     36                1022
3                 mesonbuild/meson                https://github.com/mesonbuild/meson                954                     17                 937
4                      numpy/numpy                     https://github.com/numpy/numpy                937                    148                 789
5   googleapis/google-cloud-python  https://github.com/googleapis/google-cloud-py

In [21]:
sorted_results.to_csv("swe_bench_analysis_rd1.csv", index=False)

## Re-analyzing all repos to identify the type of ground truth files extension for all swe-bench repos

In [30]:
def get_file_extension(filepath):
    # Extracts the file extension from a given path.
    if '.' in os.path.basename(filepath):
        return os.path.splitext(filepath)[1]
    return 'no_extension'

In [31]:
def analyze_swe_bench_unique_extensions(swe_bench_df):
    '''
    Finds a single unique list of all ground truth file extensions from the combined SWE-bench
    dataset dataframe:

    Args:
        swe_bench_df (pd.DataFrame): The DataFrame containing all SWE-bench data.
    '''
    print("Finding Unique Ground Truth File extensions for SWE-bench Dataset")

    if 'patch' not in swe_bench_df.columns:
        print("Error: 'patch' column not found in the DataFrame.")
        return
    
    # make a copy to avoid modifying the original dataframe
    df = swe_bench_df.copy()

    # 1. Applying the function to the 'patch' column to get a list of files for each row
    df['files_list'] = df['patch'].apply(extract_file_paths_from_patch)

    # 2. Explode the Dataframe to create one row for each file
    all_files_df = df.explode('files_list')

    # Drop any rows that became empty (if a patch had no file paths)
    all_files_df.dropna(subset=['files_list'], inplace=True)

    # 2. get the unique extension for each file using a set for automatic uniqueness
    unique_extensions = {get_file_extension(f) for f in all_files_df['files_list']}

    print("Unique File Extensions across all projects")
    print(sorted(list(unique_extensions)))



In [32]:
analyze_swe_bench_unique_extensions(swe_bench_all)

Finding Unique Ground Truth File extensions for SWE-bench Dataset
Unique File Extensions across all projects
['', '.R', '.bat', '.build', '.c', '.cfg', '.cmd', '.conf', '.css', '.css_t', '.csv', '.h', '.html', '.in', '.ini', '.ipynb', '.json', '.md', '.pip', '.ps1', '.pxd', '.pxi', '.py', '.pyi', '.pyx', '.rst', '.run', '.sh', '.svg', '.template', '.toml', '.tpl', '.txt', '.yaml', '.yml', 'no_extension']


# MetaData Extraction
Set of methods to calculate all ten metadata for selected projects from SWE-Bench

1. LOC
2. age_years
3. median_bug_year
4. num_authors
5. num_commits
6. num_dependencies
7. polyglot_index
8. bug_density
9. code_complexity
10. bug_report_verbosity

In [1]:
# Import libraries
import pandas as pd
import os
import re
import git
import subprocess
from datetime import datetime
from tqdm import tqdm
import xml.etree.ElementTree as ET
import tempfile 

## Configuration

In [2]:
# 1. Path to the input CSV file
# Must have columns: 'repo_name', 'language', 'total_unique_bug_report'
INPUT_CSV_PATH = "/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/swe_bench_metadata_input.csv"

# 2. Path to the main SWE Bench dataset file (to get commite or date info)
swe_bench_dev_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/dev-00000-of-00001.parquet'
swe_bench_train_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/train-00000-of-00001.parquet'
swe_bench_test_file_path = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/SWE-bench/data/test-00000-of-00001.parquet'

# 3. Load all split into pandas dataframe
swe_bench_dev = pd.read_parquet(swe_bench_dev_file_path)
swe_bench_train = pd.read_parquet(swe_bench_train_file_path)
swe_bench_test = pd.read_parquet(swe_bench_test_file_path)

# Combine the train, dev, test dataframes into one dataframe
swe_bench_df = pd.concat([swe_bench_train, swe_bench_dev, swe_bench_test], ignore_index=True)

# 3. Path to the parent directory where repos are cloned
CLONE_DIR = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/temp_repos/'

# 4. Path for the final output CqSV file.
OUTPUT_CSV_PATH = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/swe_bench_metadata.csv'

# 5. Extensions for Polyglot Index Calculation
LANGUAGE_EXTENSIONS = {
    'c++': ['.c', '.cc', '.cmake', '.cpp', '.cxx', '.h', '.hh', '.hpp', '.hxx', '.in', '.json', '.make', '.py', '.sh', '.xml'],
    'go': ['.go', '.json', '.proto', '.sh', '.yaml', '.yml'],
    'java': ['.gradle', '.groovy', '.java', '.json', '.properties', '.xml', '.yml', '.yaml'],
    'javascript': ['.css', '.html', '.js', '.json', '.jsx', '.mjs', '.scss', '.sh', '.ts', '.tsx', '.yaml', '.yml'],
    'kotlin': ['.gradle', '.json', '.kt', '.kts', '.properties', '.xml', '.yaml', '.yml'],
    'python': ['.bash', '.cfg', '.in', '.ini', '.json', '.py', '.sh', '.toml', '.yaml', '.yml']
}
PRIMARY_EXTENSIONS = {
    'python': ['.py'],
    'java': ['.java'],
    'kotlin': ['.kt'],
    'c++': ['.c', '.cc', '.cpp', '.cxx', '.h', '.hh', '.hpp', '.hxx'],
    'go': ['.go'],
    'javascript': ['.js', '.jsx', '.mjs', '.ts', '.tsx']
}

## Helper Methods

In [3]:
# aiming to find correct bug report date and timelines.
def get_commit_date(repo_object, commit_hash):
    '''
    Helper function to safely get a commit's authored date.
    '''
    try:
        # Get the timezone-aware datetime object
        dt_object = repo_object.commit(commit_hash).authored_datetime
        # Convert to UTC for consistency
        return dt_object.astimezone(pd.Timestamp("2000-01-01").tz)
    except Exception:
        return pd.NaT

In [4]:
# Load input files
try:
    selected_repos_df = pd.read_csv(INPUT_CSV_PATH)
    # print("len of selected repos", len(selected_repos_df))
except FileNotFoundError as e:
    print(f"Error: Input file not found. {e}")
    exit(0)

# Initialize the new column with a timezone-aware NaT
# This solves the FutureWarning by making the dtypes compatible from the start.
swe_bench_df['report_date'] = pd.to_datetime(pd.NaT, utc=True)


for _, row in tqdm(selected_repos_df.iterrows(), total=len(selected_repos_df), desc="Processing Repos for dates"):
    repo_name = row['repo_name']
    language = row['language']
    repo_path = os.path.join(CLONE_DIR, language.lower(), repo_name.replace('/', '_'))
    
    tqdm.write(f"Working on repo: {repo_name}")
    if not os.path.exists(repo_path):
        print(f"Warning: Clone repo not found for {repo_name} at {repo_path}. Skipping.")
        continue

    # Filter the main Dataframe to get the specific rows for this repo
    # Using .index is crucial for the assignment step later
    repo_data_indices = swe_bench_df[swe_bench_df['repo'] == repo_name].index

    if repo_data_indices.empty:
        print(f"Warning: No data found for {repo_name} in SWE Bench main dataframe. Skipping.")
        continue
    
    try:
        repo = git.Repo(repo_path)

        # Get the base_sha vales for the current repo's data
        base_shas = swe_bench_df.loc[repo_data_indices, 'base_commit']
        # print("len of base_shas:", len(base_shas))

        # Apply the date extraction function
        dates_for_group = base_shas.apply(lambda sha: get_commit_date(repo, sha))

        # --- FIX 1: Ensure the Series has the correct dtype before assignment ---
        # This also helps prevent the FutureWarning
        dates_for_group = pd.to_datetime(dates_for_group, utc=True)
        
        # Assign the calculated dates back to the correct rows
        # The FutureWarning should now be resolved by the initialization fix
        swe_bench_df.loc[repo_data_indices, 'report_date'] = dates_for_group

    except Exception as e:
        tqdm.write(f"Could not process repo {repo_name}. Error: {e}")

print("Pre-processing complete. 'report_date' column has been updated for selected repos")


Processing Repos for dates:   0%|          | 0/31 [00:00<?, ?it/s]/tmp/ipykernel_45186/3078671247.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<DatetimeArray>
['2013-02-22 13:59:11+00:00', '2013-12-20 13:24:20+00:00',
 '2014-01-29 16:24:30+00:00', '2014-03-22 16:37:32+00:00',
 '2014-04-08 12:33:37+00:00', '2014-10-29 17:35:20+00:00',
 '2015-02-12 22:15:45+00:00', '2015-04-13 16:33:28+00:00',
 '2015-04-29 11:48:31+00:00', '2015-06-09 00:09:15+00:00',
 ...
 '2023-03-20 12:29:36+00:00', '2023-03-23 08:26:55+00:00',
 '2023-07-04 10:25:03+00:00', '2023-06-12 17:34:40+00:00',
 '2023-06-19 19:37:52+00:00', '2023-07-11 16:53:22+00:00',
 '2023-07-18 12:17:06+00:00', '2023-08-01 15:10:51+00:00',
 '2023-08-08 22:17:58+00:00', '2023-08-19 12:58:30+00:00']
Length: 250, dtype: datetime64[ns, UTC]' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  swe_bench_df.loc

Working on repo: celery/celery
Working on repo: conan-io/conan
Working on repo: conda/conda


Processing Repos for dates:  13%|█▎        | 4/31 [00:00<00:02,  9.34it/s]

Working on repo: dagster-io/dagster
Working on repo: django/django


Processing Repos for dates:  19%|█▉        | 6/31 [00:00<00:02,  8.35it/s]

...

Working on repo: sphinx-doc/sphinx
Working on repo: sympy/sympy


Processing Repos for dates:  94%|█████████▎| 29/31 [00:04<00:00,  9.67it/s]

Working on repo: wagtail/wagtail
Working on repo: ytdl-org/youtube-dl


Processing Repos for dates: 100%|██████████| 31/31 [00:04<00:00,  7.23it/s]

Pre-processing complete. 'report_date' column has been updated for selected repos


In [5]:
# Pre-requisite: Assume 'swe_bench_df' is loaded and has the 'report_date' column
print(" Verifying the corrected 'report_date' column")

# The 'report_date' column should already be in datetime formate for your pre-processing
swe_bench_df['report_date'] = pd.to_datetime(swe_bench_df['report_date'])

# 1. Show basic statistics of the corrected dates
print("Overall corrected date statistics:")
# Filter out any NaT values for accurate stats
valid_dates = swe_bench_df['report_date'].dropna()
if not valid_dates.empty:
    print(f" Earliest Date: {valid_dates.min()}")
    print(f" Latest Date: {valid_dates.max()}")
    print(f" Median Date: {valid_dates.median()}")
else:
    print(" No valid dates found in the 'report_date' column.")

# 2. Show the distribution of bug reports by year
print("Distribution of Bug Reports per Year (top 15):")
print(valid_dates.dt.year.value_counts().sort_index(ascending=False).head(15).to_string())

# 3. Isolate and check if any '1970' dates remain
print("Checking for any remaining anomalous '1970' dates")
problematic_rows = swe_bench_df[swe_bench_df['report_date'].dt.year == 1970]

if not problematic_rows.empty:
    print(f"Warning: Found {len(problematic_rows)} entries that still have a '1970' year")
else:
    print("Success: No entries with a '1970' year were found in the 'report_date' column.")

 Verifying the corrected 'report_date' column
Overall corrected date statistics:
 Earliest Date: 2010-12-02 16:36:24+00:00
 Latest Date: 2023-08-25 20:30:21+00:00
 Median Date: 2019-06-18 19:51:53+00:00
Distribution of Bug Reports per Year (top 15):
report_date
2023    1237
2022    1700
2021    1865
2020    3598
2019    3337
2018    2662
2017    1719
2016     940
2015     989
2014    1196
2013     789
2012     168
2011      49
2010       1
Checking for any remaining anomalous '1970' dates
Success: No entries with a '1970' year were found in the 'report_date' column.


In [6]:
def count_lines_in_file(file_path):
    '''
    Counts lines in a file, handling encoding errors.
    '''
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return len(f.readlines())
    except Exception:
        return 0

In [7]:
def calculate_loc_and_polyglot(repo_path, declared_language):
    '''
    Calculates LoC and polyglot index. Auto-detects the actual primary language in the snapshot to handle
    migrations
    '''
    loc_by_lang = {lang: 0 for lang in PRIMARY_EXTENSIONS.keys()}
    loc_relevant = 0

    # first, calculate LoC for each potential primary language
    for root, _, files in os.walk(repo_path):
        for file in files:
            for lang, exts in PRIMARY_EXTENSIONS.items():
                if file.endswith(tuple(exts)):
                    loc_by_lang[lang] += count_lines_in_file(os.path.join(root, file))

    # Auto-detect the language with the most LoC in this snapshot
    actual_primary_languge = max(loc_by_lang, key=loc_by_lang.get) if loc_by_lang else declared_language
    loc_primary = loc_by_lang.get(actual_primary_languge, 0)

    # Now, calculate total relevant LoC based on the DECLARED ecosystem
    relevant_exts = tuple(LANGUAGE_EXTENSIONS.get(declared_language.lower(), []))

    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(relevant_exts):
                loc_relevant += count_lines_in_file(os.path.join(root, file))
                
    polyglot_index = (loc_primary / loc_relevant) if loc_relevant > 0 else 0
    return loc_relevant, polyglot_index

In [8]:
def count_dependencies(repo_path, language):
    """
    Calculates a proxy for dependency complexity by summing the Lines of Code (LoC)
    of standard dependency files for the primary language.
    """
    loc_count = 0
    lang = language.lower()

    dependency_files = []
    if lang == 'python':
        dependency_files = ['requirements.txt', 'pyproject.toml']
    elif lang in ['java', 'kotlin']:
        dependency_files = ['pom.xml', 'build.gradle', 'build.gradle.kts']
    elif lang == 'c++':
        dependency_files = ['CMakeLists.txt', 'Makefile'] # Example for C++
    elif lang == 'javascript':
        dependency_files = ['package.json'] # Example for JS
    elif lang == 'go':
        dependency_files = ['go.mod'] # Example for Go

    for root, _, files in os.walk(repo_path):
        for file in files:
            if file in dependency_files:
                file_path = os.path.join(root, file)
                # Simply add the number of lines in the file to the count
                loc_count += count_lines_in_file(file_path)
    
    return loc_count

In [9]:
def calculate_average_complexity(repo_path, language):
    '''
    Calculates average cyclomatic complexity using the 'lizard' tool.
    We store the ouput of lizard to a temporary file to handle large outputs reliably.
    '''
    # Create a temp file to stroe the lizard output
    with tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.txt', encoding='utf-8') as temp_out:
        temp_filename = temp_out.name

    try:
        # lizard expects 'c++' to be written as 'cpp'
        lang_for_lizard = 'cpp' if language.lower() == 'c++' else language.lower()
        # print("Language for lizard: ", lang_for_lizard)

        # Redirect stdout to the temp file
        # We run the command and tell it to write its output directly to our temp file
        result = subprocess.run(
            ['lizard', '-i', '0', repo_path], # another command ['lizard', '-l', 'lang_for_lizard', repo_path]
            stdout = open(temp_filename, 'w',  encoding='utf-8'), # Write stdout to the temp file
            stderr=subprocess.PIPE, # Still capture any errors in memory 
            check=False, text=True
        )

        # We can still manually check the return code if we want to log detailed errors
        if result.returncode != 0:
            print(f"\nWarning: Lizard finished with a non-zero exit code ({result.returncode}) for {repo_path}. This usually indicates warnings were found. Continuing to parse output.")
            # We don't return here, because the output file is likely still valid.

        # Read the results back from the file
        with open(temp_filename, 'r', encoding='utf-8') as f:
            lizard_output = f.read()

        # Get all non-empty lines from the output
        lines = [line for line in lizard_output.strip().splitlines()]

        # the summary data is on the second to last line
        if len(lines) >=3 :
            # target the line with the numbers (the last non-empty line)
            summary_line = lines[-1]

            # Split the line by whitespace
            values = summary_line.split()

            if len(values) >= 3:
                # The Avg CCN is the 3rd value (index 2)
                avg_ccn = float(values[2])
                return avg_ccn

        # Find the summary line in the output
        # If we reach here, the summary line was not found or was malformed
        print(f"\nWarning: Could not parse lizard summary for {repo_path}.")
        return 0.0
        
    except FileNotFoundError:
        # This error is critical, so we print it once and then it will return 0 for others.
        print("\nERROR: 'lizard' command not found. Please install it with 'pip install lizard'.")
        return 0.0
    except subprocess.CalledProcessError as e:
        print(f"\n Lizard command failed for {repo_path}. Stderr: {e.stderr}")
        return 0.0
    except (IndexError, ValueError) as e:
        print(f"\nFailed to extract complexity value from summary line for {repo_path}. Error: {e}")
        return 0.0
    except Exception as e:
        print(f"\nAn unexpected error occurred in calculate_average_complexity: {e}")
        return 0.0
    finally:
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

## Main Code

In [10]:
print("Starting metadata extraction for SWE-Bench Dataset...")

# Load input files
try:
    selected_repos_df = pd.read_csv(INPUT_CSV_PATH)
    # print(selected_repos_df)
except FileNotFoundError as e:
    print(f"Error: Input file not found. {e}")
    exit(0)

results = []

for _, row in tqdm(selected_repos_df.iterrows(), total=len(selected_repos_df), desc="Processing Repos"):
    repo_name = row['repo_name']
    language = row['language']
    unique_bugs = row['total_unique_bug_report']

    repo_path = os.path.join(CLONE_DIR, language.lower(), repo_name.replace('/', '_'))
    print(f"Calculating meta data for repo: {repo_name}")
    if not os.path.exists(repo_path):
        print(f"Warning: Clone repo not found for {repo_name} at {repo_path}. Skipping.")
        continue

    repo_swe_bench_data = swe_bench_df[swe_bench_df['repo'] == repo_name].copy()
    if repo_swe_bench_data.empty:
        print(f"Warning: No data found for {repo_name} in SWE Bench main dataframe. Skipping.")
        continue

    # Get the snapshot commit from the latest bug report
    latest_bug = repo_swe_bench_data.sort_values(by='report_date', ascending=False).iloc[0]
    snapshot_commit = latest_bug['base_commit']

    try:
        repo = git.Repo(repo_path)
        repo.git.checkout(snapshot_commit, f=True)

        # *****************
        # Calculate Metrics
        # *****************

        # c) Age
        # --- Metrics now use the reliable 'report_date' column ---
        min_date = repo_swe_bench_data['report_date'].min()
        max_date = repo_swe_bench_data['report_date'].max()
        age_years = ((max_date - min_date).days) / 365.25
        median_bug_year = repo_swe_bench_data['report_date'].dt.year.median()

        # d) No. of authors & e) No. of commits (Correctly scoped to the snapshot)
        all_commits = list(repo.iter_commits())
        num_commits = len(all_commits)
        num_authors = len({c.author.email for c in all_commits})

        # a) LoC & g) Polyglot Index
        loc, polyglot_index = calculate_loc_and_polyglot(repo_path, language)

        # f) No. of external dependencies
        dependencies = count_dependencies(repo_path, language)

        # h) Bug density
        kloc = loc / 1000
        bug_density = (kloc / unique_bugs) if unique_bugs > 0 else 0

        # i) code_complexity
        code_complexity = calculate_average_complexity(repo_path, language)

        # j) Calculate bug report verbosoty
        bug_report_verbosity = repo_swe_bench_data['problem_statement'].str.split().str.len().mean()
        
        results.append({
            'repo_name': repo_name,
            'language': language,
            'LoC': loc,
            'age_years': age_years,
            'median_bug_year': median_bug_year, # Raw data for later categorization
            'num_authors': num_authors,
            'num_commits': num_commits,
            'num_dependencies': dependencies,
            'polyglot_index': polyglot_index,
            'bug_density': bug_density,
            'code_complexity': code_complexity,
            'bug_report_verbosity': bug_report_verbosity
        })
    
    # except git.exec.GitCommandError as e:
    #     print(f"Error processing Git Repo {repo_name}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred for {repo_name}: {e}")

if not results:
    print("No results were generated.")
    exit(0)

# Create final DataFrame
final_df = pd.DataFrame(results)

# # b) Calculate project_size category
# final_df = final_df.groupby('language', group_keys=False).apply(categorize_by_tercile)

# Save to CSV
final_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Metdata extraction complete. Results saved to '{OUTPUT_CSV_PATH}'")


Starting metadata extraction for SWE-Bench Dataset...


Processing Repos:   0%|          | 0/31 [00:00<?, ?it/s]

Calculating meta data for repo: celery/celery


Processing Repos:   3%|▎         | 1/31 [00:01<00:53,  1.77s/it]


Calculating meta data for repo: conan-io/conan


...


Calculating meta data for repo: wagtail/wagtail


Processing Repos:  97%|█████████▋| 30/31 [02:37<00:05,  5.32s/it]


Calculating meta data for repo: ytdl-org/youtube-dl


Processing Repos: 100%|██████████| 31/31 [02:40<00:00,  5.16s/it]


Metdata extraction complete. Results saved to '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/swe_bench_metadata.csv'
